# **Реализация искуственного увеличения train датасета**
**Для решения проблемы несбалансированности данных**

In [33]:
from PIL import Image
from torchvision.transforms import transforms
import pandas as pd
from math import ceil
from tqdm import tqdm
import os

In [26]:
base_dir = "./data/dataset"
additional_dir_path = "./data/dataset/additional_train_images"
train_dir_path = "./data/dataset/train_images"

In [32]:
IMAGE_SIZE = (256, ) * 2

df = pd.read_csv(
    os.path.join(base_dir, "train_solution.csv"),
    names=["id", "label"]
)
csv_data = {
    "id": tuple(df["id"]),
    "label": tuple(df["label"])
}

all_length = len(df)
df1 = df[df["label"] == 1]
length_df1 = len(df1)
print(length_df1)

ratio = ceil(
    (all_length - length_df1) / length_df1
)

ratio

8500


5

In [28]:
transform = transforms.Compose([
    transforms.RandomResizedCrop(
        IMAGE_SIZE,
        scale=(0.8, 1.),
        ratio=(0.75, 1.33)
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.1
    )
])

In [29]:
if not os.path.exists(additional_dir_path):
    os.mkdir(
        additional_dir_path
    )

In [30]:
image_id = all_length

additional_data = {
    "id": [],
    "label": []
}
for standard_ind in df1["id"]:
    standard_image = Image.open(
        os.path.join(train_dir_path, f"{standard_ind}.jpg")
    ).convert("RGB")
    
    for _ in tqdm(range(ratio)):
        converted_image: Image.Image = transform(standard_image)
        converted_id = image_id
        converted_image.save(
            os.path.join(additional_dir_path, f"{image_id}.jpg")
        )
        additional_data["id"].append(image_id)
        additional_data["label"].append(1)
        
        image_id += 1

In [34]:
csv_data["id"] += tuple(additional_data["id"])
csv_data["label"] += tuple(additional_data["label"])

new_df = pd.DataFrame(csv_data)
new_df.to_csv(
    os.path.join("./data/dataset/", "new_train_solution.csv"),
    header=False,
    index=False
)